# 人物スポットライト動画レンダラー（Colab実行用）

スマホから **「ランタイム → すべてのセルを実行」** を1回タップするだけで、
リポジトリのクローンからダミー動画のレンダリングまでを一気に確認できます。

1. セル1：リポジトリ取得・依存関係インストール・ダミー動画の生成
2. セル2：`examples/sample.json` を `/content/out.mp4` にレンダリング
3. セル3：出力動画をノートブック内で再生し、スマホへダウンロード

それより下の「本番用（自動実行）」セクションは、Googleドライブの
`マイドライブ/spotlight_reel/` フォルダに `project.json`（と元動画）を
置いておくだけで、同じ「すべてのセルを実行」の中で自動的に動きます。
`project.json` がまだ置かれていない場合は、案内を表示して自動的に
スキップします（エラーにはなりません）。

## 1. リポジトリの取得・依存関係のインストール・ダミー動画の生成
動作確認用のダミー動画（`examples/dummy_input.mp4`）と
プロジェクトJSON（`examples/sample.json`）をその場で生成します。

In [ ]:
import os

REPO_DIR = "/content/spotlight-reel"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/digital-twin-creator/spotlight-reel.git {REPO_DIR}
else:
    print("リポジトリは既に取得済みです:", REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

# ffmpeg が無い場合のみ有効化してください（Colabには通常プリインストール済みです）
# !apt-get -y install ffmpeg

# 動作確認用のダミー動画・プロジェクトJSON・効果音・フォントを生成
!python make_dummy.py

## 2. サンプルJSONをレンダリング
`examples/sample.json`（動画パスはJSON内の指定に従う）を
`/content/out.mp4` にレンダリングします。

In [ ]:
SAMPLE_JSON_PATH = f"{REPO_DIR}/examples/sample.json"
SAMPLE_OUT_PATH = "/content/out.mp4"

!python render.py "{SAMPLE_JSON_PATH}" --out "{SAMPLE_OUT_PATH}"

print("出力ファイル:", SAMPLE_OUT_PATH)
print("サイズ:", os.path.getsize(SAMPLE_OUT_PATH), "bytes")

## 3. 出力動画の再生・スマホへのダウンロード
ノートブック内で動画を再生します。続けて `files.download` が実行され、
スマホのブラウザでダウンロードが自動的に始まります
（ダウンロードの許可を求められた場合は許可してください）。

In [ ]:
from IPython.display import Video, display

display(Video(SAMPLE_OUT_PATH, embed=True, width=360))

from google.colab import files
files.download(SAMPLE_OUT_PATH)

---
## 本番用（自動実行）

スマホ用エディタで書き出した `project.json` と元動画を、Googleドライブの
**「マイドライブ/spotlight_reel/」** フォルダに置いてから、このセクションを
実行してください（「ランタイム → すべてのセルを実行」に含めても構いません）。

- `project.json` は必ず `MyDrive/spotlight_reel/project.json` に置いてください（パス固定）。
- 動画ファイルは、まず同じ `MyDrive/spotlight_reel/` フォルダの中から
  `project.json` 内の `"video"` に書かれたファイル名で探します。
  見つからない場合は、リポジトリの `examples/`（動作確認用の `dummy_input.mp4` など）
  を探します。
- 出力は `MyDrive/spotlight_reel/output_YYYYMMDD_HHMM.mp4` として保存されます。

`project.json` がまだ置かれていない場合、このセクションは自動的にスキップされ、
案内メッセージだけが表示されます（エラーにはなりません）。

### 本番-1. Googleドライブをマウント

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Googleドライブのマウントをスキップしました（Colab以外の環境、または既にマウント済みの可能性があります）:", e)

### 本番-2. project.json を読み込み、動画を自動で探して render.py を実行
- `project.json`: `/content/drive/MyDrive/spotlight_reel/project.json` に固定
- 動画: 同じフォルダ → 見つからなければリポジトリの `examples/` の順で自動的に探します

In [ ]:
import datetime
import json as _json

DRIVE_DIR = "/content/drive/MyDrive/spotlight_reel"
JSON_PATH = os.path.join(DRIVE_DIR, "project.json")
OUT_PATH = None

if not os.path.isfile(JSON_PATH):
    print("project.json が見つかりません:", JSON_PATH)
    print("スマホ用エディタで書き出した project.json を、Googleドライブの")
    print("「マイドライブ/spotlight_reel/」フォルダに置いてから、このセルを")
    print("もう一度実行してください（本番用セクションは自動的にスキップされました）。")
else:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        _project = _json.load(f)
    video_name = _project.get("video") or ""

    candidates = [
        os.path.join(DRIVE_DIR, video_name),
        os.path.join(REPO_DIR, "examples", video_name),
    ]
    VIDEO_PATH = next((p for p in candidates if video_name and os.path.isfile(p)), None)

    if not VIDEO_PATH:
        print("動画ファイルが見つかりませんでした（video:", repr(video_name), "）")
        print("以下の場所を探しましたが見つかりませんでした:")
        for p in candidates:
            print(" -", p)
        print("動画ファイルを", DRIVE_DIR, "に置いてから、このセルをもう一度実行してください。")
    else:
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
        OUT_PATH = os.path.join(DRIVE_DIR, f"output_{timestamp}.mp4")
        print("動画:", VIDEO_PATH)
        print("JSON:", JSON_PATH)
        print("出力:", OUT_PATH)
        cmd = f'python render.py "{JSON_PATH}" --video "{VIDEO_PATH}" --out "{OUT_PATH}"'
        print(cmd)
        !{cmd}

### 本番-3. 出力の再生・スマホへのダウンロード
出力が作られていれば、ノートブック内で再生し、続けて `files.download` で
スマホへのダウンロードを開始します。

In [ ]:
if OUT_PATH and os.path.isfile(OUT_PATH):
    print("出力ファイル:", OUT_PATH)
    print("サイズ:", os.path.getsize(OUT_PATH), "bytes")
    display(Video(OUT_PATH, embed=False, width=360))
    from google.colab import files
    files.download(OUT_PATH)
else:
    print("出力ファイルがないため、再生・ダウンロードはスキップしました。")